# GenPy-LLM — Python Code Generation LLM (Kaggle Training Notebook)

This notebook trains **GenPy-LLM**, a ~300M parameter decoder-only Transformer specialized for Python code generation, built entirely from scratch using PyTorch.

**Curriculum:**
1. Install dependencies
2. Check GPU
3. Clone / set up the project
4. Download dataset
5. Train tokenizer
6. Create model & print parameter count
7. Start training (with checkpoint saving)
8. Resume training
9. Generate Python code
10. Evaluate model

## Step 1: Install Dependencies

In [ ]:
!pip install -q torch tokenizers tqdm datasets pytest

## Step 2: Check GPU

In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    props = torch.cuda.get_device_properties(0)
    print(f'VRAM: {props.total_memory / 1024**3:.2f} GB')
    print(f'BF16 supported: {torch.cuda.is_bf16_supported()}')
else:
    print('WARNING: No GPU found. Training will be very slow on CPU.')

## Step 3: Upload / Clone the GenPy-LLM Project

Upload the GenPy-LLM project zip to Kaggle and unzip it, OR clone from GitHub:

In [ ]:
import os

# Option A: If you uploaded the zip as a Kaggle dataset
# !unzip /kaggle/input/genpy-llm/genpy-llm.zip -d /kaggle/working/

# Option B: Clone from GitHub
# !git clone https://github.com/YOUR_USERNAME/GenPy-LLM.git /kaggle/working/GenPy-LLM

PROJECT_DIR = '/kaggle/working/GenPy-LLM'
os.chdir(PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')

import sys
sys.path.insert(0, PROJECT_DIR)

## Step 4: Download / Prepare Dataset

In [ ]:
# Download CodeParrot Python dataset subset (~50K examples for quick training)
!python scripts/prepare_data.py --output data/raw/python_code.jsonl --num-examples 50000

# Check file size
import os
size_mb = os.path.getsize('data/raw/python_code.jsonl') / 1024**2
print(f'Dataset size: {size_mb:.1f} MB')

# Preview a sample
import json
with open('data/raw/python_code.jsonl') as f:
    sample = json.loads(f.readline())
print('Sample:')
print(sample['text'][:300])

## Step 5: Train Tokenizer

In [ ]:
!python scripts/train_tokenizer.py \
    --data data/raw/python_code.jsonl \
    --vocab-size 32000 \
    --save-path tokenizer/genpy_tokenizer.json

In [ ]:
# Test the tokenizer
from tokenizer.tokenizer import GenPyTokenizer
tokenizer = GenPyTokenizer('tokenizer/genpy_tokenizer.json')
print(f'Vocab size: {tokenizer.vocab_size}')

test_code = 'def factorial(n):\n    if n == 0:\n        return 1'
encoded = tokenizer.encode(test_code)
decoded = tokenizer.decode(encoded)
print(f'Original:  {repr(test_code)}')
print(f'Encoded:   {encoded[:10]}...')
print(f'Decoded:   {repr(decoded)}')

## Step 6: Create Model & Print Parameter Count

In [ ]:
# Choose your config: tiny (test), medium (validation), or large (~300M)
# For Kaggle GPU training, use large:
from configs.large import config

# For quick testing:
# from configs.tiny import config

from model.genpy_llm import GenPyLLM, count_parameters

model = GenPyLLM(config)
total, trainable = count_parameters(model)

In [ ]:
# Move model to GPU
model = model.to(device)
print(f'Model on: {next(model.parameters()).device}')

## Step 7: Start Training

In [ ]:
# Full training run
# Adjust batch-size and save-every based on your GPU memory
!python scripts/pretrain.py \
    --config configs/large.py \
    --data data/raw/python_code.jsonl \
    --batch-size 4 \
    --epochs 1 \
    --save-every 500 \
    --checkpoint-dir checkpoints

## Step 8: Resume Training

If your Kaggle session was interrupted, resume from the latest checkpoint:

In [ ]:
!python scripts/pretrain.py \
    --config configs/large.py \
    --data data/raw/python_code.jsonl \
    --batch-size 4 \
    --epochs 1 \
    --save-every 500 \
    --checkpoint-dir checkpoints \
    --resume

## Step 9: Generate Python Code

In [ ]:
from genpy_llm import GenPyLLM as GenPyLLMApi

model_api = GenPyLLMApi.from_checkpoint(
    'checkpoints/latest.pt',
    tokenizer_path='tokenizer/genpy_tokenizer.json'
)

prompts = [
    'Write Python code to check if a number is odd or even.',
    'Write a Python function to compute the factorial of a number.',
    'Write a Python function to find the largest element in a list.',
]

for prompt in prompts:
    print(f'Prompt: {prompt}')
    code = model_api.generate_code(prompt, max_new_tokens=200)
    print(f'Generated code:')
    print(code)
    print('-' * 60)

## Step 10: Evaluate the Model

In [ ]:
# Quick syntax check on a few generated samples
from evaluation.syntax_check import check_syntax, syntax_pass_rate

test_prompts = [
    'Write a Python function to add two numbers.',
    'Write a Python function to reverse a string.',
    'Write a Python function to check if a number is prime.',
    'Write a Python class for a stack data structure.',
    'Write a Python function to find the factorial recursively.',
]

generated_codes = [model_api.generate_code(p, max_new_tokens=200) for p in test_prompts]
rate = syntax_pass_rate(generated_codes)
print(f'Syntax Pass Rate: {rate*100:.1f}% ({int(rate*len(test_prompts))}/{len(test_prompts)})')

for prompt, code in zip(test_prompts, generated_codes):
    ok, err = check_syntax(code)
    status = '✅' if ok else f'❌ ({err[:50]})'
    print(f'{status} | {prompt}')

In [ ]:
# Full 100-prompt benchmark (takes several minutes)
!python evaluation/benchmark.py \
    --checkpoint checkpoints/latest.pt \
    --tokenizer tokenizer/genpy_tokenizer.json \
    --max-new-tokens 200 \
    --output benchmark_results.json